# 05 — Model Training (Master Notebook)
**Goal:** Define features, perform the single stratified train/test split, train all three models (Logistic Regression, Random Forest, XGBoost) on identical training data, and export model pipelines.

---
### Workflow:
1. Load `model_ready_dataset.csv`.
2. Define canonical features and encode target `engagement_level`.
3. Perform **ONE** Stratified 80/20 train/test split.
4. Save `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`.
5. Build preprocessing pipelines (`OneHotEncoder` + scaling/passthrough).
6. Train Logistic Regression, Random Forest, and XGBoost.
7. Save all 3 model pipelines to `../ml/saved_models/`.

In [2]:
%pip install xgboost
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load Model-Ready Dataset

In [3]:
df = pd.read_csv('../data/processed/model_ready_dataset.csv')
print(f"Loaded dataset shape: {df.shape}")

categorical_features = ['Platform', 'Content_Type', 'Category', 'Day_of_Week', 'Sentiment', 'Influencer_Tier']
numerical_features = ['Hour_of_Day', 'Month', 'Hashtag_Count', 'Content_Length', 'Follower_Count', 'Has_Media', 'Is_Verified']
feature_cols = categorical_features + numerical_features

X = df[feature_cols].copy()
y = df['engagement_level'].copy()

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
print(f"Target classes: {label_encoder.classes_} -> {label_encoder.transform(label_encoder.classes_)}")

Loaded dataset shape: (5000, 15)
Target classes: ['High' 'Low' 'Medium'] -> [0 1 2]


## 2. Perform Single Stratified Train/Test Split (80/20)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print(f"X_train shape: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test shape:  {X_test.shape} | y_test:  {y_test.shape}")

# Save the split files to data/processed/ so teammates and evaluation notebook can use them
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
pd.DataFrame({'engagement_level': y_train}).to_csv('../data/processed/y_train.csv', index=False)
pd.DataFrame({'engagement_level': y_test}).to_csv('../data/processed/y_test.csv', index=False)

os.makedirs('../ml/saved_models', exist_ok=True)
joblib.dump(label_encoder, '../ml/saved_models/label_encoder.pkl')
print("Successfully saved split CSVs and label_encoder.pkl!")

X_train shape: (4000, 13) | y_train: (4000,)
X_test shape:  (1000, 13) | y_test:  (1000,)
Successfully saved split CSVs and label_encoder.pkl!


## 3. Define Preprocessors & Pipelines for All 3 Models

In [5]:
# Preprocessor for Logistic Regression (with StandardScaler)
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ]
)

# Preprocessor for Tree Models (with passthrough)
preprocessor_unscaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numerical_features)
    ]
)

# Define the 3 pipelines
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor_scaled),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42, C=1.0))
    ]),
    'Random Forest': Pipeline([
        ('preprocessor', preprocessor_unscaled),
        ('classifier', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
    ]),
    'XGBoost': Pipeline([
        ('preprocessor', preprocessor_unscaled),
        ('classifier', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42, eval_metric='mlogloss', n_jobs=-1))
    ])
}

## 4. 5-Fold Stratified Cross-Validation

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_summary = {}

for name, pipe in models.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1_macro')
    cv_summary[name] = scores
    print(f"{name:<20} | 5-Fold CV Macro F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

Logistic Regression  | 5-Fold CV Macro F1: 0.7115 (+/- 0.0119)
Random Forest        | 5-Fold CV Macro F1: 0.7067 (+/- 0.0068)
XGBoost              | 5-Fold CV Macro F1: 0.7040 (+/- 0.0156)


## 5. Fit All 3 Models & Save Pipelines

In [7]:
file_map = {
    'Logistic Regression': '../ml/saved_models/logistic_regression_pipeline.pkl',
    'Random Forest': '../ml/saved_models/random_forest_pipeline.pkl',
    'XGBoost': '../ml/saved_models/xgboost_pipeline.pkl'
}

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, file_map[name])
    print(f"Trained and saved: {name} -> {file_map[name]}")

Trained and saved: Logistic Regression -> ../ml/saved_models/logistic_regression_pipeline.pkl
Trained and saved: Random Forest -> ../ml/saved_models/random_forest_pipeline.pkl
Trained and saved: XGBoost -> ../ml/saved_models/xgboost_pipeline.pkl
